# 1. CLONE REPO

In [ ]:
%cd /kaggle/working
!git clone https://github.com/PhuongThao-2005/LViT.git

%cd /kaggle/working/LViT
!git checkout BTRXD-LViT-T-SlidingInference

# 2. COPY DATASET FULLSIZE

In [ ]:
!cp -r /kaggle/input/datasets/phuongthao205/btrxd-fullsize/BTRXD_fullsize /kaggle/working/LViT/datasets/

# 3. CHECK DATASET STRUCTURE

In [ ]:
import os

print(os.listdir("./datasets"))

print(os.listdir("./datasets/BTRXD_fullsize"))

print(os.listdir("./datasets/BTRXD_fullsize/Train_Folder"))

print(os.listdir("./datasets/BTRXD_fullsize/Train_Folder/img")[:5])

print(os.listdir("./datasets/BTRXD_fullsize/Train_Folder/labelcol")[:5])

# 4. VISUALIZE MASK

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

mask_dir = "./datasets/BTRXD_fullsize/Train_Folder/labelcol"

sample = os.listdir(mask_dir)[0]

print(sample)

mask = Image.open(os.path.join(mask_dir, sample))

print(mask.size)
print(mask.mode)

plt.imshow(mask, cmap="gray")
plt.show()

# 5. COPY CHECKPOINT

In [ ]:
import shutil, os

SESSION = "DEBUG_05.12_14h44"

src = '/kaggle/input/notebooks/lehngoc/lvit-t-100-label/BTRXD_tumor_l100/LViT/DEBUG_05.12_14h44/models/best_model.pth.tar'

dst_dir = f'/kaggle/working/BTRXD_fullsize/LViT/{SESSION}/models/'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, dst_dir + 'best_model.pth.tar')
print("Checkpoint copied to:", dst_dir)

# 6. SET TEST_SESSION VÀO CONFIG.PY

In [ ]:
%%writefile /kaggle/working/LViT/Config.py
# -*- coding: utf-8 -*-
import os
import torch
import time
import ml_collections

save_model = True
tensorboard = True
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
use_cuda = torch.cuda.is_available()
seed = 666
os.environ['PYTHONHASHSEED'] = str(seed)

cosineLR = True
n_channels = 3
n_labels = 1
epochs = 200
img_size = 224
print_frequency = 50
save_frequency = 10
vis_frequency = 50
early_stopping_patience = 100
pretrain = False

task_name = 'BTRXD_fullsize'        # ← đổi từ BTRXD_tumor_l100 sang BTRXD_fullsize

learning_rate = 3e-4
batch_size = 4
accumulation_steps = 2

model_name = 'LViT'

train_dataset = './datasets/' + task_name + '/Train_Folder/'
val_dataset   = './datasets/' + task_name + '/Val_Folder/'
test_dataset  = './datasets/' + task_name + '/Test_Folder/'
task_dataset  = './datasets/' + task_name + '/Train_Folder/'

label_plan_csv = './datasets/' + task_name + '/label_plan_100.csv'

session_name = 'DEBUG_' + time.strftime('%m.%d_%Hh%M')

base_save_dir = '/kaggle/working/'

save_path          = os.path.join(base_save_dir, task_name, model_name, session_name) + os.sep
model_path         = os.path.join(save_path, 'models') + os.sep
tensorboard_folder = os.path.join(save_path, 'tensorboard_logs') + os.sep
logger_path        = os.path.join(save_path, session_name + '.log')
visualize_path     = os.path.join(save_path, 'visualize_val') + os.sep

def get_CTranS_config():
    config = ml_collections.ConfigDict()
    config.transformer = ml_collections.ConfigDict()
    config.KV_size = 960
    config.transformer.num_heads = 4
    config.transformer.num_layers = 4
    config.expand_ratio = 4
    config.transformer.embeddings_dropout_rate = 0.1
    config.transformer.attention_dropout_rate = 0.1
    config.transformer.dropout_rate = 0
    config.patch_sizes = [16, 8, 4, 2]
    config.base_channel = 64
    config.n_classes = 1
    return config

# Phase 2A — test session từ Phase 1
test_session = "DEBUG_05.12_14h44"

# 7. INSTALL PACKAGES

In [33]:
!pip install -q openpyxl tensorboardX thop transformers ml-collections scipy scikit-learn

# 8. RUN SLIDING WINDOW REFERENCE

In [ ]:
!python test_sliding_window.py

# 9. SHOW RESULT

In [ ]:
result = f'/kaggle/working/BTRXD_tumor_l100/LViT/{SESSION}/sliding_window_test/results.txt'
print(open(result).read())